# Sepsis calibration: reproduce the recorded study
Upload **Sepsis_Complete_GitHub_Code.zip** from this conversation. This notebook
reproduces the alert-policy analysis from frozen predictions, with an optional
external evaluation. It does not perform clinical validation or repeat full
primary training. Python 3.12 is the verified environment. The original
single-stage notebooks are historical and expect their earlier archive names.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, subprocess, sys
uploaded = files.upload()
archive = Path(next(name for name in uploaded if name == "Sepsis_Complete_GitHub_Code.zip"))
base = Path("/content/sepsis_reproduction")
base.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (base / name).resolve().is_relative_to(base.resolve()):
            raise ValueError("Unsafe archive path")
    z.extractall(base)
project = base / "Sepsis_Alert_Calibration_GitHub"
assert (project / "README.md").exists()
print(sys.version)
print(project)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements_complete.txt"], cwd=project, check=True)
subprocess.run([sys.executable, "verify_repository.py"], cwd=project, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "stage2/tests", "stage3/tests", "stage4/tests"], cwd=project, check=True)


In [ ]:
subprocess.run([sys.executable, "reproduce.py", "--stage", "followup"], cwd=project, check=True)
files.download(str(project / "reproductions/followup/verification.json"))


## Optional: reproduce both external evaluations
The next cell acquires the exact public demo inputs, verifies checksums and
compares both original and amended runs with the saved results. Models remain
frozen. It takes longer than the follow-up-only cells. Saved original outputs
and locks are not overwritten.


In [ ]:
RUN_EXTERNAL = False
if RUN_EXTERNAL:
    subprocess.run([sys.executable, "download_external_data.py"], cwd=project, check=True)
    subprocess.run([sys.executable, "verify_repository.py", "--with-data"], cwd=project, check=True)
    subprocess.run([sys.executable, "reproduce.py", "--stage", "external"], cwd=project, check=True)
    files.download(str(project / "reproductions/external/verification.json"))
else:
    print("Set RUN_EXTERNAL = True to acquire public inputs and verify both runs.")
